# 

# 3D Finger EDU

In this notebook we will simulate a real system: a [3D finger robot](https://github.com/open-dynamic-robot-initiative/open_robot_actuator_hardware/tree/master/mechanics/finger_edu_v1).

<img src="https://github.com/open-dynamic-robot-initiative/open_robot_actuator_hardware/raw/master/mechanics/finger_edu_v1/images/finger_edu_1.jpg" alt="drawing" width="200"/>

The goal is to use what you learned in the previous two notebooks in order to:
- make the robot arm hit a box with its end effector
- analyse the contact

### Loading the robot in MuJoCo

The robot description is available [here](https://gitlab.laas.fr/gepetto/example-robot-data/-/tree/master/robots/finger_edu_description). However, the description provided is a URDF description, which is not directly compatible with MuJoCo.

For practical reason, the description has already been converted to a MuJoCo compatible format (see hints on how to do it [here](https://mujoco.readthedocs.io/en/stable/modeling.html#urdf-extensions)).
Have a look at this [file](../robot_description/finger_edu_description/xml/finger_edu_scene.xml) to see how a floor has been added to the seen. We used `<include file="finger_edu.xml"/>` to import the robot model (no floor) without having to copy the full file.

In [ ]:
MODEL_FILE = "../robot_description/finger_edu_description/xml/finger_edu_scene.xml"

In [ ]:
# Import and same functions as in the last notebooks
import mujoco
import time
import mujoco.viewer
MAX_SIM_TIME = 30 #s

def  sim_viewer(model,
				data,
				controller = None,
				max_sim_time=MAX_SIM_TIME):
	
	use_controller = True
	
	def key_callback(keycode):
		if chr(keycode) == ' ':
			nonlocal use_controller
			use_controller = not use_controller
			
	with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
		
		# Visualize joints
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

		# Close the viewer automatically after 30 wall-seconds.
		start = time.time()
		while viewer.is_running() and time.time() - start < max_sim_time:
			step_start = time.time()
			
			# Control loop
			if controller != None:
				torque = np.zeros(model.nv)
				if use_controller:
					q, v = data.qpos, data.qvel
					torque = controller.get_torques(q, v)
				n_act = model.nu
				data.ctrl = torque[:n_act] # TODO

			# mj_step can be replaced with code that also evaluates
			# a policy and applies a control signal before stepping the physics.
			mujoco.mj_step(model, data)

			# Pick up changes to the physics state, apply perturbations, update options from GUI.
			viewer.sync()

			# Rudimentary time keeping, will drift relative to wall clock.
			time_until_next_step = model.opt.timestep - (time.time() - step_start)
			if time_until_next_step > 0:
				time.sleep(time_until_next_step)

import numpy as np
from typing import Union

class PDController():
	def __init__(self,
					kp: Union[float, np.ndarray] = 1.,
					kd: Union[float, np.ndarray] = 1.
					) -> None:
		self.kp = kp
		self.kd = kd
		self.n_act = 1
		
	def set_command(self, q_des: np.ndarray) -> None:
		self.q_des = q_des
		if isinstance(self.q_des, np.ndarray) or isinstance(q_des, list):
			self.n_act = len(self.q_des)
		
	def get_torques(self, q: np.ndarray, v: np.ndarray) -> np.ndarray:
		torques = self.kp * (self.q_des - q[:self.n_act]) + self.kd * (np.zeros_like(v) - v)[:self.n_act]
		return torques

Let's visualize the robot in the viewer:

In [ ]:
model = mujoco.MjModel.from_xml_path(MODEL_FILE)
data = mujoco.MjData(model)

sim_viewer(model, data, max_sim_time=MAX_SIM_TIME)

### Control and plotting

In this section we will apply the simple PD controller seen in the first notebook and apply it to the 3D finger robot.

**Questions**:
- Fill the `Q_DES` values so that the robot last link swings at 90 deg (parallel to the floor).

In [ ]:
model = mujoco.MjModel.from_xml_path(MODEL_FILE)
data = mujoco.MjData(model)

# TODO: Set the desired joint position [q1, q2, q3] in radians
Q_DES = [] # rad
KP = 25.
KD = 0.05
controller = PDController(KP, KD)
controller.set_command(Q_DES)
sim_viewer(model, data, controller)

There are quite some oscillations in the motion. Let's try to plot the joint trajectories in order to better tune the gains.

**Questions**:
- Save the correct values in the `sim_data` dict at each simulation steps in the code below

In [ ]:
from collections import defaultdict

def sim_viewer( model,
				data,
				controller = None,
				max_sim_time=30):
	
	# Collect data for plotting
	sim_data = defaultdict(list)
	use_controller = True
	
	def key_callback(keycode):
		if chr(keycode) == ' ':
			nonlocal use_controller
			use_controller = not use_controller
			
	with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
		
		# Visualize joints
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

		# Close the viewer automatically after 30 wall-seconds.
		start = time.time()
		while viewer.is_running() and time.time() - start < max_sim_time:
			step_start = time.time()
			
			# Control loop
			if controller != None:
				torque = np.zeros(model.nv)
				if use_controller:
					q, v = data.qpos, data.qvel
					torque = controller.get_torques(q, v)
				data.ctrl = torque # TODO
			
			# TODO: save data for plotting
			sim_data['qpos'].append(None)
			sim_data['qvel'].append(None)
			sim_data['ctrl'].append(None)
			sim_data['time'].append(None)

			# mj_step can be replaced with code that also evaluates
			# a policy and applies a control signal before stepping the physics.
			mujoco.mj_step(model, data)

			# Pick up changes to the physics state, apply perturbations, update options from GUI.
			viewer.sync()

			# Rudimentary time keeping, will drift relative to wall clock.
			time_until_next_step = model.opt.timestep - (time.time() - step_start)
			if time_until_next_step > 0:
				time.sleep(time_until_next_step)

	return sim_data

# Here is a function to plot the simulation data
import matplotlib.pyplot as plt

def plot_sim_data(sim_data):
	time = np.array(sim_data['time'])
	qpos = np.array(sim_data['qpos'])
	qvel = np.array(sim_data['qvel'])
	ctrl = np.array(sim_data['ctrl'])
	print(f"Simulation time: {time[-1]:.2f} seconds")
	print(qpos.shape, qvel.shape, ctrl.shape)

	fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
	
	axs[0].plot(time, qpos)
	axs[0].set_ylabel('Position (qpos)')
	axs[0].set_title('Joint Positions')
	
	axs[1].plot(time, qvel)
	axs[1].set_ylabel('Velocity (qvel)')
	axs[1].set_title('Joint Velocities')
	
	axs[2].plot(time, ctrl)
	axs[2].set_ylabel('Control (ctrl)')
	axs[2].set_xlabel('Time (s)')
	axs[2].set_title('Control Inputs')
	
	plt.tight_layout()
	plt.show()

In [ ]:
model = mujoco.MjModel.from_xml_path(MODEL_FILE)
data = mujoco.MjData(model)

Q_DES = [0., 0., -np.pi/2] # rad
KP = 25.
KD = 0.05
controller = PDController(KP, KD)
controller.set_command(Q_DES)
sim_data = sim_viewer(model, data, controller)
# Plot data
plot_sim_data(sim_data)

**Questions**:
- Tune the `KD` gain to reduce the oscillations of the motion.

In [ ]:
model = mujoco.MjModel.from_xml_path(MODEL_FILE)
data = mujoco.MjData(model)

Q_DES = [0., 0., -np.pi/2] # rad
KP = 1.
# TODO: Tune the derivative gain KD
KD = 0.
controller = PDController(KP, KD)
controller.set_command(Q_DES)
sim_data = sim_viewer(model, data, controller, max_sim_time=2)
plot_sim_data(sim_data)

### Visualizing the contact force between the cube and the finger

In this section, our goal is to make the 3d finger hit the cube and plot the contact force exerced over time.

We first have to add a cube to our model.

**Questions**:
To add the cube to your MuJoCo model:

- Create a new file at `"../robot_description/finger_edu_description/xml/finger_edu_scene_cube.xml"` and copy `finger_edu_scene.xml`
- Add a body name `box` at position `0.1 -0.05 0.0 5`
- Add a free joint
- Add a `box` geom to the body of mass `0.01` and size `0.05 0.05 0.05` (you can choose the color).

Simulate with the cube:

In [ ]:
MODEL_FILE = "../robot_description/finger_edu_description/xml/finger_edu_scene_cube.xml"

model = mujoco.MjModel.from_xml_path(MODEL_FILE)
data = mujoco.MjData(model)

Q_DES = [0., 0., -np.pi/2,] # rad
KP = 1.
KD = 0.07
controller = PDController(KP, KD)
controller.set_command(Q_DES)
sim_data = sim_viewer(model, data, controller, max_sim_time=30)

Let's plot the contact data.
The issue is that the cube is in contact with both the floor and the robot. Therefore, contact pairs with the floor need to be filtered out. 

**Questions**:
- First find the id `floor_id` of the floor geometry.

In [ ]:
floor_geom_name = 'floor'
# TODO: get the geom id of the floor
floor_id = None
print(f"Floor geom id: {floor_id}")

We need to modify the `plot_contact` function from last notebook to only consider contacts involving the robot.
`discard_contacts_id` is a new argument containing all contact geometries that will be filtered out.

**Questions**:
- Modify the function to check if at least one of the two contact geoms are in `discard_contacts_id`

In [ ]:
def plot_contacts(model,
				  data,
				  controller=None,
				  discard_contacts_id = [],
				  max_sim_time=1.4):
	n_steps = int(max_sim_time / model.opt.timestep)

	# allocate
	sim_time = np.zeros(n_steps)
	ncon = np.zeros(n_steps)
	force = np.zeros((n_steps,3))    # Sum of all contact forces
	penetration = np.zeros(n_steps)  # Penetration distance of the contact
	forcetorque = np.zeros(6)

	with mujoco.viewer.launch_passive(model, data) as viewer:
	
		# Visualize contact
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True
		
		# tweak scales of contact visualization elements
		model.vis.scale.contactwidth = 0.1
		model.vis.scale.contactheight = 0.03
		model.vis.scale.forcewidth = 0.05
		model.vis.map.force = 0.3
		
		# Close the viewer automatically after 30 wall-seconds.
		i = 0
		start = time.time()
		while viewer.is_running() and i < n_steps:
			step_start = time.time()
			
			# Control loop
			if controller != None:
				torque = np.zeros(model.nv)
				q, v = data.qpos, data.qvel
				torque = controller.get_torques(q, v)
				n_act = model.nu
				data.ctrl = torque[:n_act] # TODO


			# mj_step can be replaced with code that also evaluates
			# a policy and applies a control signal before stepping the physics.
			mujoco.mj_step(model, data)

			# Pick up changes to the physics state, apply perturbations, update options from GUI.
			viewer.sync()

			# Rudimentary time keeping, will drift relative to wall clock.
			time_until_next_step = model.opt.timestep - (time.time() - step_start)
			if time_until_next_step > 0:
				time.sleep(time_until_next_step)
				
			# iterate over active contacts, save force and distance
			for j, cnt in enumerate(data.contact):

				# TODO: check if contact geoms are in discard_contacts_id
				if None:
					continue

				mujoco.mj_contactForce(model, data, j, forcetorque)
				force[i] += forcetorque[0:3]
				pen_dist = cnt.dist
				penetration[i] = min(penetration[i], pen_dist)
				
			# Fill data arrays
			sim_time[i] = data.time
			ncon[i] = data.ncon
			i += 1
			
	# plot
	_, ax = plt.subplots(2, 2, sharex=True, figsize=(10, 10))

	lines = ax[0,0].plot(sim_time, force)
	ax[0,0].set_title('contact force')
	ax[0,0].set_ylabel('Newton')
	ax[0,0].legend(iter(lines), ('normal z', 'x', 'y'));

	ax[0,1].plot(sim_time, ncon)
	ax[0,1].set_title('number of contacts')
	ax[0,1].set_yticks(range(6))

	ax[1,0].plot(sim_time, force[:,0])
	ax[1,0].set_title('normal (z) force')
	ax[1,0].set_ylabel('Newton')
	ax[1,0].legend()

	ax[1,1].plot(sim_time, 1000*penetration)
	ax[1,1].set_title('penetration depth')
	ax[1,1].set_ylabel('millimeter')
	ax[1,1].set_xlabel('second')

	plt.tight_layout()
	plt.show()

In [ ]:
model = mujoco.MjModel.from_xml_path(MODEL_FILE)
data = mujoco.MjData(model)

Q_DES = [0., 0., -np.pi/2,] # rad
KP = 1.
KD = 0.07
controller = PDController(KP, KD)
controller.set_command(Q_DES)
plot_contacts(model, data, controller=controller, discard_contacts_id=[floor_id], max_sim_time=0.25)

We can now plot the contact data of the contact between the robot and the box.

**Questions**:
- Analyse the plot to estimate the maximum penetration distance and maximum force exerted on the object.